# 03 · Video perturbations (as an attack)
Paper protocol: accuracy = 1 − fraction of initially-correct frames flipped (10 noise draws); R = area under accuracy vs magnitude, bootstrapped over trajectories (10; fog 20); LERS = R / log10(FLOPs). Noise is matched to each network's input size.

In [ ]:
import json, torch, numpy as np
from flynet_robustness import config
from flynet_robustness.models import get_flynet
from flynet_robustness.feature_extraction import load_cnn_network
from flynet_robustness.decoders import load_decoder
from flynet_robustness.metrics import bootstrap_robustness, lers
device = 'cuda' if torch.cuda.is_available() else 'cpu'
network_id = 'FlyNet'   # any registry id: 'CTRNN', 'sCTRNN', 'flynet_degree_matched_0', 'vCNN_5_frames', 'RAFT_12', 'RAFT_12_S', ...
if config.is_cnn(network_id):
    net = load_cnn_network(network_id, device=device)
else:
    model1, model2, idxs1, idxs2 = get_flynet(network_id, device=device)
    net = (model1, idxs1, model2, idxs2)
decoder = load_decoder(network_id, device=device)
trajectories = json.load(open('data/configs/attack_trajectories.json'))   # accepted hit trajectories
from flynet_robustness.evaluation import evaluate_perturbation, PERTURBATIONS
print(list(PERTURBATIONS))

In [ ]:
name = 'AWN_frames'   # AWN_texture, AWN_frames, Shot_noise, Flicker, Motion_jitter, Fog
magnitudes = PERTURBATIONS[name][3]
acc = np.array([evaluate_perturbation(net, decoder, network_id, name,
                                      base_kwargs=dict(config.DEFAULT_STIMULUS, **t, device=device), device=device)['accuracy']
                for t in trajectories[:20 if name == 'Fog' else 10]]).T      # (magnitudes, trajectories)
R, R_std = bootstrap_robustness(magnitudes, acc)
print(f'{name}: R = {R:.3f} ± {R_std:.3f}   LERS = {lers(R, network_id):.3f} ± {lers(R_std, network_id):.3f}')

In [ ]:
from flynet_robustness.plotting import plot_accuracy_curve
plot_accuracy_curve(magnitudes, {network_id: acc.mean(1)}, xlabel=config.PERTURBATION_AXIS[name][0],
                    errors={network_id: acc.std(1) / np.sqrt(acc.shape[1])});